# 01 — ReAct Agent Demo

Demonstrates the **ReAct (Reason + Act)** agent loop with built-in tools and the **Console** for clean interactive output.

The agent:
1. Receives a user query
2. Reasons about which tools to call
3. Executes tools and observes results
4. Iterates until it can produce a final answer

**Prerequisites**: `OPENAI_API_KEY` set in the environment.

In [1]:
import os
# Uncomment and fill in if not set in your environment
# os.environ['OPENAI_API_KEY'] = 'sk-...'
print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

OPENAI_API_KEY set: True


## Imports

The `Console` class gives us clean, coloured output — no noisy JSON logs.

In [2]:
from raavan.core.agents.react_agent import ReActAgent
from raavan.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from raavan.providers.llm.openai.openai_client import OpenAIClient
from raavan.core.memory.unbounded_memory import UnboundedMemory
from raavan.core.context.implementations import UnboundedContext
from raavan.core.console import Console

## Build the agent

In [3]:
tools = [CalculatorTool(), GetCurrentTimeTool()]

agent = ReActAgent(
    name='DemoBot',
    description='A helpful assistant for demonstration.',
    model_client=OpenAIClient(model='gpt-4o'),
    tools=tools,
    memory=UnboundedMemory(),
    model_context=UnboundedContext(),
    max_iterations=5,
    verbose=True,
)

console = Console(agent)
print(f"Agent '{agent.name}' ready with {len(tools)} tools.")

Agent 'DemoBot' ready with 2 tools.


## Single-shot run (non-streaming)

`console.run()` pretty-prints tool calls and the final answer.

In [4]:
result = await console.run(
    'What is the square root of 256 multiplied by 14? Also what time is it?'
)

You → What is the square root of 256 multiplied by 14? Also what time is it?

╭──────────────────────────────────────────────────── DemoBot ────────────────────────────────────────────────────╮
│                                                                                                                 │
│  The square root of 256 multiplied by 14 is 224.0.                                                              │
│                                                                                                                 │
│  The current time in UTC is 18:46 on March 15, 2026.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

completed · 3 steps · 3 tool calls · 892 tokens · 6.3s

## Streaming run

`console.run_stream()` prints tokens as they arrive.

In [5]:
# Reset memory so we start fresh
agent.reset()
result = await console.run_stream('Compute 15 factorial and tell me the current time.')

You → Compute 15 factorial and tell me the current time.

✖ calculator

{
  "error": "invalid syntax (<string>, line 1)",
  "expression": "15!"
}

✔ get_current_time

{
  "datetime": "2026-03-15T18:47:11.805715",
  "timezone": "UTC",
  "timestamp": 1773580631.805715
}

...

,

202

6

.

3 tool calls · 4.7s

## Interactive mode

`console.interactive()` gives you a chat REPL. Type `/help` for commands, `exit` to quit.

In [ ]:
# Uncomment to start interactive chat (type 'exit' to quit)
# agent.reset()
# await console.interactive()